In [3]:
import pandas as pd

raw = pd.read_csv("../raw_data/anand_vihar/Raw_Data_2025_site_301_anand_vihar_delhi_dpcc_1Hr.csv")
raw.head()

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2025-01-01 00:00:00,251.0,374.0,102.03,58.83,114.27,74.83,13.20,2.81,1.93,...,NaN,11.10,89.00,0.33,251.0,NaN,0.0,5.0,740.0,NaN
1,2025-01-01 01:00:00,269.0,437.0,102.10,61.60,115.80,74.32,12.35,2.74,3.05,...,NaN,10.85,89.75,0.35,251.0,NaN,0.0,5.0,740.0,NaN
2,2025-01-01 02:00:00,238.0,326.0,59.90,63.22,82.35,81.50,14.50,2.38,4.08,...,NaN,11.08,89.25,0.68,251.0,NaN,0.0,5.0,740.0,NaN
3,2025-01-01 03:00:00,227.0,324.0,44.08,61.25,68.40,70.03,16.45,2.00,2.50,...,NaN,11.40,86.50,1.05,251.0,NaN,0.0,5.0,740.0,NaN
4,2025-01-01 04:00:00,286.0,381.0,31.90,51.35,53.25,66.12,15.75,1.81,2.55,...,NaN,11.42,85.50,0.60,251.0,NaN,0.0,5.0,740.0,NaN


In [4]:
import glob

stations = ["anand_vihar","rohini","rk_puram","punjabi_bagh","ito","okhla"]
station_dfs = []
for station in stations:
    files = glob.glob(f"../raw_data/{station}/*.csv")
    dfs = [pd.read_csv(f) for f in files]
    for d in dfs:
        d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='mixed', dayfirst=True)
    sdf = pd.concat(dfs).sort_values('Timestamp').reset_index(drop=True)
    station_dfs.append(sdf)
all_stations = pd.concat(station_dfs).reset_index(drop=True)
all_stations.shape

(210384, 25)

In [5]:
raw.columns

Index(['Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
       'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)',
       'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)',
       'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)',
       'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)',
       'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)'],
      dtype='object')

In [6]:
raw.shape


(8760, 25)

In [7]:
raw.isna().sum()

Timestamp                 0
PM2.5 (µg/m³)           616
PM10 (µg/m³)            608
NO (µg/m³)              581
NO2 (µg/m³)             471
NOx (ppb)               488
NH3 (µg/m³)             448
SO2 (µg/m³)            1066
CO (mg/m³)              466
Ozone (µg/m³)           675
Benzene (µg/m³)         622
Toluene (µg/m³)         622
Xylene (µg/m³)         8760
O Xylene (µg/m³)       8760
Eth-Benzene (µg/m³)    8760
MP-Xylene (µg/m³)      8760
AT (°C)                 394
RH (%)                  400
WS (m/s)                410
WD (deg)                458
RF (mm)                8760
TOT-RF (mm)               0
SR (W/mt2)              400
BP (mmHg)               400
VWS (m/s)              8760
dtype: int64

In [8]:
raw = raw.drop(columns=['Xylene (µg/m³)','O Xylene (µg/m³)','Eth-Benzene (µg/m³)'])

In [9]:
raw['Timestamp'] = pd.to_datetime(
    raw['Timestamp'],
    format='mixed',
    dayfirst=True
)


In [10]:
raw = raw.set_index('Timestamp')


In [11]:
aqi_jan = pd.read_excel('../aqi_target/2025/AQI_Data_City_Level_Delhi_January_2025_Hourly.xlsx')


In [12]:
aqi_long = aqi_jan.melt(id_vars='Date', var_name='Hour', value_name='AQI')

In [13]:
aqi_long["Timestamp"] = pd.to_datetime("2025-01-" + aqi_long["Date"].astype(str).str.zfill(2) + " " + aqi_long["Hour"], format="%Y-%m-%d %H:%M")

In [14]:
aqi_long.head()

,Date,Hour,AQI,Timestamp
0,1,00:00,314.0,2025-01-01
1,2,00:00,320.0,2025-01-02
2,3,00:00,331.0,2025-01-03
3,4,00:00,383.0,2025-01-04
4,5,00:00,379.0,2025-01-05


In [15]:
months = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
all_months = []
for num, name in months.items():
    df = pd.read_excel(f"../aqi_target/2025/AQI_Data_City_Level_Delhi_{name}_2025_Hourly.xlsx")
    long = df.melt(id_vars='Date', var_name='Hour', value_name='AQI')
    long['Timestamp'] = pd.to_datetime(f"2025-{num:02d}-" + long['Date'].astype(str).str.zfill(2) + " " + long['Hour'], format="%Y-%m-%d %H:%M")
    long = long.drop(columns=['Date','Hour'])
    all_months.append(long)
aqi_full_year = pd.concat(all_months).sort_values('Timestamp').reset_index(drop=True)

In [16]:
aqi_full_year.shape

(8760, 2)

In [17]:
merged = pd.merge(raw, aqi_full_year.set_index("Timestamp"), left_index=True, right_index=True, how="inner")
merged.shape

(8760, 22)

In [18]:
merged.head(
)

,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),...,AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s),AQI
Timestamp,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,251.0,374.0,102.03,58.83,114.27,74.83,13.20,2.81,1.93,6.23,...,11.10,89.00,0.33,251.0,NaN,0.0,5.0,740.0,NaN,314.0
2025-01-01 01:00:00,269.0,437.0,102.10,61.60,115.80,74.32,12.35,2.74,3.05,6.25,...,10.85,89.75,0.35,251.0,NaN,0.0,5.0,740.0,NaN,320.0
2025-01-01 02:00:00,238.0,326.0,59.90,63.22,82.35,81.50,14.50,2.38,4.08,5.72,...,11.08,89.25,0.68,251.0,NaN,0.0,5.0,740.0,NaN,324.0
2025-01-01 03:00:00,227.0,324.0,44.08,61.25,68.40,70.03,16.45,2.00,2.50,6.00,...,11.40,86.50,1.05,251.0,NaN,0.0,5.0,740.0,NaN,328.0
2025-01-01 04:00:00,286.0,381.0,31.90,51.35,53.25,66.12,15.75,1.81,2.55,5.72,...,11.42,85.50,0.60,251.0,NaN,0.0,5.0,740.0,NaN,328.0


In [19]:
merged.to_csv("../processed_data/anand_vihar_2025_merged.csv")

In [20]:
years = [2022, 2023, 2024, 2025]
all_data = []
for year in years:
    for num, name in months.items():
        df = pd.read_excel(f"../aqi_target/{year}/AQI_Data_City_Level_Delhi_{name}_{year}_Hourly.xlsx")
        long = df.melt(id_vars='Date', var_name='Hour', value_name='AQI')
        hour_clean = long['Hour'].astype(str).str[:5]
        long['Timestamp'] = pd.to_datetime(f"{year}-{num:02d}-" + long['Date'].astype(str).str.zfill(2) + " " + hour_clean, format="%Y-%m-%d %H:%M")
        long = long.drop(columns=['Date','Hour'])
        all_data.append(long)
aqi_all_years = pd.concat(all_data).sort_values('Timestamp').reset_index(drop=True)
aqi_all_years.shape

(35040, 2)

In [21]:
years = [2022, 2023, 2024, 2025]
raw_list = []
for year in years:
    df = pd.read_csv(f"../raw_data/anand_vihar/Raw_Data_{year}_site_301_anand_vihar_delhi_dpcc_1Hr.csv")
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='mixed', dayfirst=True)
    raw_list.append(df)
raw_all_years = pd.concat(raw_list).sort_values('Timestamp').reset_index(drop=True)
raw_all_years.shape

(35064, 25)

In [22]:
raw_all_years = raw_all_years.set_index('Timestamp')
aqi_all_years_idx = aqi_all_years.set_index('Timestamp')
merged_all = pd.merge(raw_all_years, aqi_all_years_idx, left_index=True, right_index=True, how='inner')
merged_all.shape

(35040, 25)

In [24]:
import glob

stations = ["anand_vihar","rohini","rk_puram","punjabi_bagh","ito","okhla"]
station_dfs = []
for station in stations:
    files = glob.glob(f"../raw_data/{station}/*.csv")
    dfs = [pd.read_csv(f) for f in files]
    for d in dfs:
        d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='mixed', dayfirst=True)
    sdf = pd.concat(dfs).sort_values('Timestamp').reset_index(drop=True)
    station_dfs.append(sdf)
all_stations = pd.concat(station_dfs).reset_index(drop=True)
all_stations.shape

(210384, 25)

In [25]:
numeric_cols = all_stations.select_dtypes(include='number').columns
city_avg = all_stations.groupby('Timestamp')[numeric_cols].mean().reset_index()
city_avg.shape

(35064, 25)

In [28]:
final = pd.merge(city_avg.set_index('Timestamp'), aqi_all_years_idx, left_index=True, right_index=True, how='inner')
final.shape
final.head()

,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),...,AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s),AQI
Timestamp,,,,,,,,,,,,,,,,,,,,,
2022-01-01 00:00:00,423.166667,496.350000,176.578333,57.991667,182.230000,51.575000,20.536667,2.806,7.100000,9.4600,...,9.502,81.080,0.380,173.104,0.0,0.0,5.764,936.652,0.04,330.0
2022-01-01 01:00:00,432.458333,555.458333,135.956667,50.976667,144.071667,56.878333,16.146667,2.316,7.004000,8.7600,...,9.188,82.606,0.306,194.034,0.0,0.0,5.746,936.574,0.00,331.0
2022-01-01 02:00:00,392.125000,517.708333,116.920000,51.358333,126.503333,48.664000,11.786667,1.716,7.193333,7.7950,...,9.286,82.586,0.376,218.206,0.0,0.0,5.726,936.446,0.00,334.0
2022-01-01 03:00:00,375.291667,478.666667,102.205000,50.961667,114.271667,51.765000,10.565000,1.372,7.221667,6.7575,...,9.088,83.538,0.394,212.150,0.0,0.0,5.744,936.374,0.00,337.0
2022-01-01 04:00:00,324.458333,425.416667,82.060000,49.823333,96.921667,54.168000,12.603333,1.060,7.483333,7.5220,...,8.842,83.906,0.378,170.410,0.0,0.0,5.792,936.234,0.00,340.0


In [27]:
final.to_csv("../processed_data/delhi_6station_avg_2022_2025.csv")